# Evaluate explanations: conventional vs. list-aware fidelity metrics
Main experiment of the paper. For each test user, the target $t$ is the top-1 recommendation. For $K_e = 1..\pi$, the $K_e$ most important history items of an explanation are removed (or kept alone), and we compare the recommendation list before and after.

**Conventional (target-only) metrics** (Barkan et al., 2024): `POS_at_n`, `DEL`, `INS`, `CDCG`, and their non-target counterparts `NONTARGET_POS_at_n` and `NONTARGET_CDCG` (the same quantity averaged over the other items of the top-$K$ list).

**List-aware metrics** (this paper): an explanation is credited only if it affects the target more than the rest of the list.
| Metric | Refines | Columns |
|---|---|---|
| Counterfactual Validity | POS | `COV_CV` (target leaves top-$K$ **and** RBO of the remaining list $\geq \tau$), `LS`, `LS_SUCC` |
| Necessity | DEL | `COV_NEC`, `DELTA_NEC` |
| Sufficiency | INS | `COV_SUF`, `DELTA_SUF` |
| Rank Degradation | CDCG | `COV_RD`, `DELTA_RD`, `ODR_T`, `ODR_NT` |

Hyperparameters come from `config.py`: $K=20$, $\pi=10$, $\tau=0.7$, $c=5$, RBO $p=0.9$.

Output: `results/<dataset>/<recommender>_<explainer>_metrics.pkl`, a dict `{metric: array over K_e = 1..π}`.

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import pickle
import warnings

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
import ipynb

from config import *

In [ ]:
data_name = "ML1M"          # ML1M, Yahoo, Pinterest
recommender_name = "MLP"    # MLP (= MF in the paper), VAE
explainers_to_eval = EXPLAINERS   # jaccard, cosine, lime, shap, accent, spinrec, lxr

files_path = dataset_dir(data_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

output_type = OUTPUT_TYPE[recommender_name]
num_items = NUM_ITEMS[data_name]
hidden_dim = HIDDEN_DIM[(data_name, recommender_name)]
recommender_path = CHECKPOINTS_DIR / RECOMMENDER_CHECKPOINTS[(data_name, recommender_name)]

## Data and recommender

In [ ]:
test_data = pd.read_csv(files_path / f'test_data_{data_name}.csv', index_col=0)
static_test_data = pd.read_csv(files_path / f'static_test_data_{data_name}.csv', index_col=0)
test_array = static_test_data.iloc[:,:-2].to_numpy()
items_array = np.eye(num_items)
all_items_tensor = torch.Tensor(items_array).to(device)

kw_dict = {'device':device,
          'num_items': num_items,
          'num_features': num_items,
          'demographic':False,
          'all_items_tensor':all_items_tensor,
          'static_test_data':static_test_data,
          'items_array':items_array,
          'output_type':output_type,
          'recommender_name':recommender_name}

In [ ]:
from ipynb.fs.defs.help_functions import *
from ipynb.fs.defs.recommenders_architecture import *

In [ ]:
def load_recommender():
    if recommender_name == 'MLP':
        recommender = MLP(hidden_dim, **kw_dict)
    elif recommender_name == 'VAE':
        if data_name == "Pinterest":
            recommender = VAE(PINTEREST_VAE_CONFIG, **kw_dict)
        else:
            recommender = VAE(VAE_CONFIG, **kw_dict)
    else:
        raise ValueError(f"Unknown recommender name: {recommender_name}")

    recommender_checkpoint = torch.load(recommender_path, map_location=device)
    recommender.load_state_dict(recommender_checkpoint, strict=False)
    recommender.eval()
    for param in recommender.parameters():
        param.requires_grad = False
    return recommender

recommender = load_recommender()

## Metrics

In [ ]:
METRICS = [
    # Conventional (target only) and their non-target counterparts
    'DEL', 'INS', 'POS_at_5', 'POS_at_10', 'POS_at_20', 'CDCG',
    'NONTARGET_POS_at_5', 'NONTARGET_POS_at_10', 'NONTARGET_POS_at_20', 'NONTARGET_CDCG',
    # Counterfactual Validity
    'COV_CV', 'LS', 'LS_SUCC',
    # Necessity, Sufficiency, Rank Degradation
    'COV_NEC', 'DELTA_NEC',
    'COV_SUF', 'DELTA_SUF',
    'COV_RD', 'DELTA_RD', 'ODR_T', 'ODR_NT',
]
# DEL, INS              target score after removing / keeping only the explanation
# POS_at_n              1 if the target left the top-n after removal
# CDCG                  1 / log2(rank_after + 1) of the target (lower = better)
# NONTARGET_POS_at_n    share of non-target top-K items that dropped by at least n positions
# NONTARGET_CDCG        mean 1 / log2(max(rank shift, 0) + 2) over non-target top-K items
# COV_CV                target left the top-K AND LS >= TAU
# LS                    RBO between the top-K lists (target excluded) before and after removal
# LS_SUCC               LS, only for users whose target left the top-K (NaN otherwise)
# ODR_T, ODR_NT         offset-discounted rank change of the target / mean over non-targets
# COV_x                 1 if the target is affected more than the mean non-target item
# DELTA_x               mean non-target effect minus target effect (higher = better)

In [ ]:
# ====================================================================== #
#  Helpers                                                                #
# ====================================================================== #
def rbo_similarity(list1, list2, p=RBO_P):
    """Extrapolated RBO (Webber et al., 2010, eq. 32). 1 = identical, 0 = disjoint."""
    if not list1 or not list2:
        return 0.0
    S, L = (list1, list2) if len(list1) <= len(list2) else (list2, list1)
    s, l = len(S), len(L)
    seen_S, seen_L, overlap = set(), set(), 0
    X = [0] * (l + 1)
    for d in range(1, l + 1):
        if d <= s:
            if S[d - 1] in seen_L:
                overlap += 1
            seen_S.add(S[d - 1])
        if L[d - 1] in seen_S:
            overlap += 1
        seen_L.add(L[d - 1])
        X[d] = overlap
    sum1 = sum(X[d] / d * p**d for d in range(1, l + 1))
    sum2 = sum(X[s] * (d - s) / (s * d) * p**d for d in range(s + 1, l + 1))
    return (1 - p) / p * (sum1 + sum2) + ((X[l] - X[s]) / l + X[s] / s) * p**l


def odr(rank_before, rank_after, c):
    """Offset-discounted rank. 1 = unchanged, < 1 = dropped, > 1 = moved up."""
    return np.log2(rank_before + c) / np.log2(rank_after + c)


def odr_cdcg(rank_before, rank_after, c):
    """CDCG-style discount of the rank shift: 1 / log2(max(rank_after - rank_before, 0) + 2)."""
    return 1 / np.log2(max((rank_after - rank_before), 0)+2)
    
def ratio(new, old, eps=1e-8):
    """new / old; NaN if old is ~0."""
    return new / old if abs(old) > eps else np.nan


def ranks(scored):
    """{item: score} sorted by score -> {item: 1-based rank}."""
    return {it: r + 1 for r, it in enumerate(scored)}

In [ ]:
# ====================================================================== #
#  Per-user metrics                                                       #
# ====================================================================== #
def single_user_metrics(user_tensor, t, expl, model, num_steps, tau=TAU, c=C, **kw):
    """
    user_tensor : user history with the target t already set to 0
    expl        : list of (item, importance), most important first
    Returns {metric: array of length num_steps}, one value per K_e = 1..num_steps.
    """
    orig = get_top_k(user_tensor, user_tensor, model, **kw)
    orig_items, orig_rank = list(orig), ranks(orig)

    non_targets = [x for x in orig_items[:K] if x != t]       # N_u = L_K(x_u) \ {t}
    list_wo_t   = [x for x in orig_items if x != t][:K]       # L_K^{-t}(x_u)
    expl_sorted = sorted(expl.items() if hasattr(expl, 'items') else expl,
                         key=lambda e: e[1], reverse=True)
    expl_items = [int(e[0]) for e in expl_sorted if user_tensor[int(e[0])] > 0]   # history items only

    out = {m: np.full(num_steps, np.nan) for m in METRICS}

    for i in range(num_steps):
        k = i + 1
        mask = torch.zeros_like(user_tensor)
        mask[torch.tensor(expl_items[:k], dtype=torch.long)] = 1

        rem = get_top_k(user_tensor * (1 - mask), user_tensor, model, **kw)   # explanation removed
        kep = get_top_k(user_tensor * mask,       user_tensor, model, **kw)   # only explanation kept
        rem_items, rem_rank = list(rem), ranks(rem)
        rank_t = rem_rank[t]

        # --- Conventional (target only) ---
        out['DEL'][i] = rem[t]
        out['INS'][i] = kep[t]
        for n in (5, 10, 20):
            out[f'POS_at_{n}'][i] = float(rank_t > n)
            # share of non-target items that dropped by at least n positions
            out[f'NONTARGET_POS_at_{n}'][i] = np.mean(
                [rem_rank[j] - orig_rank[j] >= n for j in non_targets])
        out['CDCG'][i] = odr_cdcg(1, rank_t, 1)
        out['NONTARGET_CDCG'][i] = np.mean([odr_cdcg(orig_rank[j], rem_rank[j], 5) for j in non_targets])
        # --- Counterfactual Validity ---
        ls = rbo_similarity(list_wo_t, [x for x in rem_items if x != t][:K])
        out['LS'][i] = ls
        if rank_t > K:
            out['LS_SUCC'][i] = ls
        out['COV_CV'][i] = float(rank_t > K and ls >= tau)

        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)   # nanmean of an all-NaN list

            # --- Necessity: share of score left after removal ---
            d_t   = ratio(rem[t], orig[t])
            d_bar = np.nanmean([ratio(rem[j], orig[j]) for j in non_targets])
            out['COV_NEC'][i]   = float(d_t < d_bar)
            out['DELTA_NEC'][i] = d_bar - d_t

            # --- Sufficiency: share of score kept with only the explanation ---
            r_t   = ratio(kep[t], orig[t])
            r_bar = np.nanmean([ratio(kep[j], orig[j]) for j in non_targets])
            out['COV_SUF'][i]   = float(r_t > r_bar)
            out['DELTA_SUF'][i] = r_t - r_bar

        o_t   = odr(orig_rank[t], rank_t, c)
        o_bar = np.mean([odr(orig_rank[j], rem_rank[j], c) for j in non_targets])
        out['ODR_T'][i]    = o_t
        out['ODR_NT'][i]   = o_bar
        out['COV_RD'][i]   = float(o_t < o_bar)
        out['DELTA_RD'][i] = o_bar - o_t

    return out

In [ ]:
# ====================================================================== #
#  Evaluation over all users                                              #
# ====================================================================== #
def eval_one_expl_type(expl_name, num_steps=PI, tau=TAU, c=C):
    print(f' ============ {data_name} {recommender_name} explained by {expl_name} ============')

    with open(expl_dict_path(data_name, recommender_name, expl_name), 'rb') as f:
        expl_dict = pickle.load(f)

    recommender.eval()
    per_user = {m: [] for m in METRICS}

    with torch.no_grad():
        for i in tqdm(range(test_array.shape[0])):
            user_tensor = torch.FloatTensor(test_array[i].copy()).to(device)
            user_id = int(test_data.index[i])

            t = int(get_user_recommended_item(user_tensor, recommender, **kw_dict).detach().cpu().numpy())
            user_tensor[t] = 0

            res = single_user_metrics(user_tensor, t, expl_dict[user_id], recommender,
                                      num_steps, tau=tau, c=c, **kw_dict)
            for m in METRICS:
                per_user[m].append(res[m])

    # Mean over users, ignoring NaN (e.g., LS_SUCC for users whose target stayed in the top-K)
    results = {}
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        for m in METRICS:
            results[m] = np.nanmean(np.stack(per_user[m]), axis=0)
    results['NUM_USERS'] = test_array.shape[0]

    for m in METRICS:
        print(f'{m:>10}_{expl_name}: K_e={num_steps}: {results[m][-1]:.4f}   '
              f'per K_e: {np.round(results[m], 4)}')

    return results


def check_scores(n_users=3):
    """Run once: get_top_k scores should equal recommender_run scores."""
    with torch.no_grad():
        for i in range(n_users):
            ut = torch.FloatTensor(test_array[i].copy()).to(device)
            t = int(get_user_recommended_item(ut, recommender, **kw_dict).detach().cpu().numpy())
            ut[t] = 0
            it = torch.FloatTensor(items_array[t]).to(device)
            single = float(recommender_run(ut, recommender, it, t, **kw_dict).detach().cpu().numpy())
            vector = get_top_k(ut, ut, recommender, **kw_dict)[t]
            print(f'user {i}: recommender_run={single:.6f}  get_top_k={vector:.6f}  diff={abs(single - vector):.2e}')

In [ ]:
# Sanity check (optional): get_top_k scores should match recommender_run scores
check_scores()

## Run all explainers and save

In [ ]:
out_dir = RESULTS_DIR / data_name
out_dir.mkdir(parents=True, exist_ok=True)

for expl_name in explainers_to_eval:
    if not expl_dict_path(data_name, recommender_name, expl_name).exists():
        print(f'skipping {expl_name}: {expl_dict_path(data_name, recommender_name, expl_name)} not found')
        continue
    metrics = eval_one_expl_type(expl_name, num_steps=PI, tau=TAU, c=C)
    with open(metrics_path(data_name, recommender_name, expl_name), 'wb') as f:
        pickle.dump(metrics, f)